# TD02 — Réseau mono-couche (3 perceptrons)

## Objectif

Passer d'un neurone (TD01) à une **couche de 3 perceptrons** : chaque neurone $k$ calcule un score $z_k = W_k \cdot x + b_k$, et le réseau renvoie l'indice du score maximal $y = \mathrm{argmax}(z_1, z_2, ..., z_n)$. On l'apprend avec la **règle de Rosenblatt multi-classes**, puis on l'applique en 2D et en 20D (lettres S/V/X).


**Notes - Écart au modèle du cours**
1. Dans le cours, on associait un facteur `-1` au biais. Ici on associe un facteur `1` ( donc on a la sortie `z = W \cdot x + b` au lieu de `z = W \cdot x - b`). La seule différence est au niveau de l'implémentation, le réseau de neurones se comporte exactement de la même manière.
2. Dans le cours, la fonction d'activation (ex. Heaviside) s'applique à la sortie d'**un seul** neurone : $y = H(w \cdot x + b)$. Ici, on regroupe $K$ neurones et la décision se fait par un `argmax` appliqué au **vecteur** des scores, c'est-à-dire à la sortie de **plusieurs** neurones : $y = \mathrm{argmax}(W x + b)$.


**Exemple**: entrée de dimension deux, avec une couche de 3 neurones:
![image.png](assets/02_mono_couche-image-2.png)

## Préambule : l'opérateur `argmax`

`argmax` renvoie l'**indice** (la position) de la plus grande valeur d'un vecteur — pas la valeur elle-même :

**Exemples**
- $\mathrm{argmax}([0.2,\; -0.5,\; 0.7]) = 2$ (le maximum $0.7$ est en position 2)
- $\mathrm{argmax}([-1.0,\; 3.0,\; 2.0]) = 1$
- $\mathrm{argmax}([0.0,\; 0.0,\; 0.0]) = 0$ (en cas d'égalité, `np.argmax` renvoie le premier indice)

En Python : `np.argmax([0.2, -0.5, 0.7])` renvoie `2`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants
VecInt = np.ndarray[tuple[int], np.dtype[np.int64]]       # vecteur 1D d'entiers

## 1. Étude en 2D

Premier cas d'étude, en dimension 2 : trois classes gaussiennes (jeu 1 séparable, jeu 2 piégé au centre).



### 1.1 Génération de jeux de données 2D

On génère deux jeux de données: `(X1, t1)` et `(X2, t2)`

In [ ]:
def gen_jeu1(n=25, sigma=0.6, seed=0):
    rng = np.random.default_rng(seed)
    centres = [(-2, -2), (2, -2), (0, 2)]
    X_list, t_list = [], []
    for k, (cx, cy) in enumerate(centres):
        pts = rng.normal(loc=(cx, cy), scale=sigma, size=(n, 2))
        X_list.append(pts)
        t_list += [k] * n
    return np.vstack(X_list), np.array(t_list)


def gen_jeu2(n=15, seed=1):
    rng = np.random.default_rng(seed)
    Xa = rng.normal(loc=(0, 0), scale=0.6, size=(n, 2))
    Xb = rng.normal(loc=(0, 0), scale=0.6, size=(n, 2))
    X = np.vstack([Xa, Xb])
    t = np.array([0] * n + [1] * n)
    return X, t

In [ ]:
X1, t1 = gen_jeu1()
_X2, _t2 = gen_jeu2()
X2 = np.vstack([X1, _X2])
t2 = np.concatenate([t1, _t2])

### 1.2 Visualisation des données

> **Exercice 1** — Les cellules ci-dessous affichent les deux jeux : `X1, t1` (jeu 1) et `X2, t2` (jeu 2). Comparez-les : qu'est-ce qui les distingue ?

In [ ]:
def display_data(X, t, titre=None):
    couleurs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
    fig, ax = plt.subplots()
    for k in sorted(set(t)):
        ax.scatter(
            X[t == k, 0], X[t == k, 1],
            c=couleurs[k % len(couleurs)],
            edgecolors='k', s=25, label=f"classe {k}"
        )
    ax.set_aspect('equal')
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    if titre:
        ax.set_title(f"{titre} ({len(X)} pts)")
    ax.legend()
    ax.grid(True)
    plt.show()

In [ ]:
display_data(X1, t1, titre="Jeu 1 : séparable")
display_data(X2, t2, titre="Jeu 2 : piège central")

### 1.3 Modèle : couche mono-couche `y = argmax(W·x + b)`

Une couche de $K$ perceptrons : $W$ de forme $(K \times d)$, $b$ de forme $(K,)$. La règle de Rosenblatt multi-classes : si la prédiction $p$ diffère de la cible $t$, on **renforce** le bon neurone $t$ (faux négatif) et on **punit** le mauvais gagnant $p$ (faux positif).

> **Exercice 2** — Complétez `__init__(d, K)` : initialise `W` (forme `(K, d)`) et `b` (forme `(K,)`) à zéro. Puis complétez `predict(x)` : calcule `h = W·x + b` et renvoie l'indice du score maximal (`np.argmax`).

> **Exercice 3** — Complétez `learn_one(x, t, r=0.1)` : si la prédiction `p` diffère de la cible `t`, on **renforce** le bon neurone `t` et on **punit** le mauvais gagnant `p`.

> **Exercice 4** — Complétez `train(X, t, epochs=50)` : boucle sur les époques, compte les erreurs, s'arrête dès que le réseau classe tout correctement.

In [ ]:
class ReseauMonoCouche:
    def __init__(self, d: int, K: int):
        # BEGIN
        # raise NotImplementedError
        ## Indice : W de forme (K, d) et b de forme (K,), initialisés à zéro
        self.W = np.zeros((K, d))
        self.b = np.zeros(K)
        # END

    def predict(self, x: Vec) -> int:
        # BEGIN
        # raise NotImplementedError
        h = self.W @ x + self.b
        return int(np.argmax(h))
        # END

    def learn_one(self, x: Vec, t: int, r: float = 0.1) -> None:
        # BEGIN
        # raise NotImplementedError
        ## Indice : p = self.predict(x) ; si p != t : W[t]+=r*x, b[t]+=r, W[p]-=r*x, b[p]-=r
        p = self.predict(x)
        if p != t:
            self.W[t] += r * x
            self.b[t] += r
            self.W[p] -= r * x
            self.b[p] -= r
        # END

    def train(self, X: Mat, t: VecInt, epochs: int = 50) -> 'ReseauMonoCouche':
        # BEGIN
        # raise NotImplementedError
        ## Indice : pour chaque époque, compte les erreurs et arrête si err == 0
        for _ in range(epochs):
            err = 0
            for xi, ti in zip(X, t):
                if self.predict(xi) != int(ti):
                    err += 1
                self.learn_one(xi, int(ti))
            if err == 0:
                break
        return self
        # END

#### Vérification de `predict`

On teste le calcul `h = W·x + b` et l'`argmax` sur des scores choisis.

In [ ]:
def check_predict():
    c = ReseauMonoCouche(d=2, K=3)
    c.W = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
    c.b = np.zeros(3)
    assert c.predict(np.array([1.0, 0.0])) == 0
    assert c.predict(np.array([0.0, 1.0])) == 1
    assert c.predict(np.array([-1.0, -1.0])) == 2
    print("predict check — OK")


check_predict()

#### Vérification de `learn_one`

Une étape d'apprentissage doit exactement renforcer `t` et punir `p`.

In [ ]:
def check_learn_one():
    c = ReseauMonoCouche(d=2, K=3)
    # predict([1,1]) = argmax([0,0,0]) = 0 ; cible 1 -> erreur
    c.learn_one(np.array([1.0, 1.0]), 1, r=0.1)
    assert np.array_equal(c.W[1], np.array([0.1, 0.1])), c.W[1]
    assert c.b[1] == 0.1, c.b[1]
    assert np.array_equal(c.W[0], np.array([-0.1, -0.1])), c.W[0]
    assert c.b[0] == -0.1, c.b[0]
    assert np.array_equal(c.W[2], np.array([0.0, 0.0]))
    print("learn_one check — OK")


check_learn_one()

#### Vérification de `train`

`train` doit apprendre le jeu 1 (séparable) à 100 %.

In [ ]:
def check_train():
    c = ReseauMonoCouche(d=2, K=3)
    c.train(X1, t1)
    preds = np.array([c.predict(x) for x in X1])
    assert np.array_equal(preds, t1), "train n'a pas appris le jeu 1"
    print("train check — OK")


check_train()

### 1.4 Entraînement : jeu 1 vs jeu 2

> **Exercice 5** — Complétez `accuracy` (pourcentage de bonnes prédictions), puis entraînez deux couches identiques : `couche1` sur `X1` et `couche2` sur `X2`. Comparez les accuracies : que se passe-t-il ?

In [ ]:
def accuracy(reseau: ReseauMonoCouche, X: Mat, t: VecInt) -> float:
    """Pourcentage de bonnes prédictions (sur les données d'entraînement)."""
    # BEGIN
    # raise NotImplementedError
    pred = np.array([reseau.predict(x) for x in X])
    return (pred == t).mean() * 100
    # END

In [ ]:
def check_accuracy():
    c = ReseauMonoCouche(d=2, K=3)
    c.W = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
    c.b = np.zeros(3)
    X = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
    t = np.array([0, 1, 2])
    assert accuracy(c, X, t) == 100.0
    print("accuracy check — OK")


check_accuracy()

In [ ]:
couche1 = ReseauMonoCouche(d=2, K=3)
couche1.train(X1, t1)
acc_jeu_1 = accuracy(couche1, X1, t1)
print(f"accuracy jeu 1 : {acc_jeu_1:.1f}%")
couche2 = ReseauMonoCouche(d=2, K=3)
couche2.train(X2, t2)
acc_jeu_2 = accuracy(couche2, X2, t2)
print(f"accuracy jeu 2 : {acc_jeu_2:.1f}%")

#### Frontières de décision

`plot_frontieres` affiche :
- **fond coloré** = vraie décision `argmax_k h_k` (régions convexes, bords = `h_i == h_j`) ;
- **pointillés noirs** = zéros de chaque score `h_k = 0` (où le neurone `k` bascule). **Ce N'EST PAS la frontière de décision !**

>  Lisez la figure :
> 1. Trouvez un pointillé qui traverse une zone de couleur uniforme. Que concluez-vous ?
> 2. Comparez les résultats jeu 1 vs jeu 2.

In [ ]:
def plot_frontieres(ax, X, t, couche, lim=(-4, 4)):
    xx, yy = np.meshgrid(np.linspace(lim[0], lim[1], 300), np.linspace(lim[0], lim[1], 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = np.argmax(grid @ couche.W.T + couche.b, axis=1).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5],
                colors=['tab:blue', 'tab:orange', 'tab:green'], alpha=0.12)
    for k, c in enumerate(['tab:blue', 'tab:orange', 'tab:green']):
        ax.scatter(X[t == k, 0], X[t == k, 1], c=c, edgecolors='k', s=25, label=f"classe {k}")
    for k in range(3):
        H = (grid @ couche.W[k] + couche.b[k]).reshape(xx.shape)
        ax.contour(xx, yy, H, levels=[0], colors='k', linewidths=1.5, linestyles='--')
    ax.plot([], [], 'k--', label='h_k=0 (indicatif, pas frontière)')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid(True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_frontieres(axes[0], X1, t1, couche1)
axes[0].set_title("Jeu 1")
plot_frontieres(axes[1], X2, t2, couche2)
axes[1].set_title("Jeu 2")
plt.tight_layout()
plt.show()

## 2. Étude en 20D : lettres S/V/X en 5×4 (20 dimensions)

Même classe `ReseauMonoCouche`, réutilisée en dimension 20 : 3 templates `TPL` encodés en ±1, bruités par `n_flip` pixels inversés. Objectif : montrer que le même modèle fonctionne en 20D quand les données sont séparables, et lire chaque ligne de `W` comme un prototype.

### 2.1 Génération des lettres bruitées

3 templates `TPL` encodés en ±1, bruités en inversant `n_flip` pixels au hasard.

In [ ]:
TPL = {
    0: np.array([0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0]),
    1: np.array([1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0]),
    2: np.array([1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1]),
}


def enc(v):
    return np.where(v == 1, 1.0, -1.0)


def gen_lettres(n_per=10, n_flip=2, seed=2):
    rng = np.random.default_rng(seed)
    X, t = [], []
    for k in range(3):
        base = enc(TPL[k])
        for _ in range(n_per):
            w = base.copy()
            idx = rng.choice(20, size=n_flip, replace=False)
            w[idx] *= -1
            X.append(w)
            t.append(k)
    return np.array(X), np.array(t)

In [ ]:
def affiche_lettre(lettre: Vec, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.imshow(lettre.reshape(5, 4), cmap="RdBu", vmin=-1, vmax=1)


affiche_lettre(TPL[0])

> **Exercice 6** — Entraînez la couche en 20D sur des lettres bruitées à 2 flips (2.2), testez sa robustesse à 5 flips (2.3), puis vérifiez une lettre à la main (2.4).

### 2.2 Entraînement (2 flips)

On entraîne avec un peu de bruit (2 flips) pour obtenir une résistance au bruit.

In [ ]:
X_let, t_let = gen_lettres()  # 30 lettres : 10 par classe, bruitées à 2 flips

# BEGIN
# raise NotImplementedError
## Indice : c_let = ReseauMonoCouche(d=20, K=3) puis c_let.train(X_let, t_let)
c_let = ReseauMonoCouche(d=20, K=3)
c_let.train(X_let, t_let)
pred_let = np.array([c_let.predict(x) for x in X_let])
print(f"accuracy sur le train : {(pred_let == t_let).mean() * 100:.1f}%")
# END

### 2.3 Test à 5 flips

Déterminez l'accuracy du réseau sur des lettres **nouvelles**, bruitées à 5 flips.

In [ ]:
# BEGIN
# raise NotImplementedError
## Indice : X_let_test, t_let_test = gen_lettres(n_per=10, n_flip=5, seed=99) puis comparez les prédictions
X_let_test, t_let_test = gen_lettres(n_per=10, n_flip=5, seed=99)
pred_test = np.array([c_let.predict(x) for x in X_let_test])
print(f"accuracy lettres test (5 flips) : {(pred_test == t_let_test).mean() * 100:.1f}%")
# END

### 2.4 Vérification à la main

Choisissez une lettre de test, affichez-la, et vérifiez que le réseau l'a reconnue en regardant les scores $h = W \cdot x + b$ de chaque neurone (le plus grand `h_k` donne la prédiction).

In [ ]:
# BEGIN
# raise NotImplementedError
## Indice : h = c_let.W @ X_let_test[0] + c_let.b ; le plus grand h_k est la prédiction
h = c_let.W @ X_let_test[0] + c_let.b
print("scores :", h)
print("prédiction :", int(np.argmax(h)))
affiche_lettre(X_let_test[0])
# END

### 2.5 Lire `W` : quel prototype pour quelle lettre ?

Chaque ligne de `W` (20 nombres) ré-affichée en 5×4 (`RdBu`, rouge = +1, bleu = −1) est un prototype appris.

> **Exercice 7** — Affichez les templates et les poids appris comme des images : associez chaque ligne de `W` à sa lettre. Que remarque-t-on ?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8, 5))

# BEGIN
# raise NotImplementedError
## Indice : enc(TPL[k]) en haut, c_let.W[k] en bas
for k in range(3):
    axes[0, k].imshow(enc(TPL[k]).reshape(5, 4), cmap="RdBu", vmin=-1, vmax=1)
    axes[0, k].set_title(f"template {'SVX'[k]}")
    axes[0, k].axis("off")
    axes[1, k].imshow(c_let.W[k].reshape(5, 4), cmap="RdBu", vmin=-1, vmax=1)
    axes[1, k].set_title(f"W classe {k} -> lettre ?")
    axes[1, k].axis("off")
plt.suptitle("haut = templates, bas = poids appris (associez !)")
plt.tight_layout()
plt.show()
# END

### 2.6 Robustesse au bruit

On fait varier `n_flip` de 0 à 8 (train fixe à 2 flips) et on trace l'accuracy du test.

> **Exercice 8** — Tracez la courbe `accuracy(n_flip)`. À partir de quel `n_flip` passe-t-on sous 80 % ? Pourquoi ? (distance au prototype vs bruit).

In [ ]:
# BEGIN
# raise NotImplementedError
## Indice : pour nf de 0 à 8, générez le test, mesurez l'accuracy, puis tracez
flips = list(range(0, 9))
accs = []
for nf in flips:
    Xt, tt = gen_lettres(n_per=10, n_flip=nf, seed=100 + nf)
    pt = np.array([c_let.predict(x) for x in Xt])
    accs.append((pt == tt).mean() * 100)
fig, ax = plt.subplots()
ax.plot(flips, accs, marker='o')
ax.set_xlabel("n_flip (pixels inversés)")
ax.set_ylabel("accuracy test %")
ax.set_title("Robustesse au bruit (train flip=2)")
ax.grid(True)
plt.show()
print(list(zip(flips, [f'{a:.1f}%' for a in accs])))
# END

## Synthèse

> **Exercice 9** — Pourquoi le même modèle échoue sur le jeu 2 (2D) mais réussit en 20D sur les lettres ?

> **Réponse attendue**
> Ce n'est pas la dimension qui compte, c'est la **séparabilité linéaire**. Les lettres forment trois nuages bien séparés autour des prototypes (±1 + quelques flips) → Rosenblatt converge. Le jeu 2 a ses classes 0/1 **mélangées au centre** → aucun `W` linéaire ne marche, il faut du non-linéaire / multi-couches (TD03).



## Conclusion
- Séparable → Rosenblatt converge, régions convexes (fond `argmax`).
- `h_k = 0` (pointillés) ≠ frontière : la frontière est `h_i == h_j`.
- Piège central → oscillation, < 100 % : limite du linéaire.
- Lettres 20D → OK car séparables ; `W` lisible = prototypes ; robustesse chute avec le bruit.
- Suite : besoin de non-linéarité / multi-couches (TD03).